# 03 — Bridge: from toy environments to a real-ish RPG battle

Nothing fundamental changes when we move from LineWorld to Pong to Elder Mist.

What changes is how hard it is to represent the situation and predict consequences.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Same loop, richer worlds</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">LINEWORLD</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">PONG</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">ATARI</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">ELDER MIST</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">LIFEOS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
from pathlib import Path
import sys, matplotlib.pyplot as plt
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))
from visuals import pipeline

fig = pipeline(
    ["observation","state rep","policy","action","environment","reward"],
    title="The invariant RL skeleton"
)
plt.show()

## Complexity grows in different places

LineWorld has trivial observations and transitions.

Elder Mist adds delayed effects, resource management, an active Sword, charged moves, and lock-breaking.

LifeOS adds hidden external facts, messy telemetry, humans, and long horizons.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Where difficulty comes from</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SIMPLE STATE</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">DELAYED CONSEQUENCES</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">PARTIAL OBSERVABILITY</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">REAL WORLD</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
worlds = ["LineWorld","Toy Pong","Atari","Elder Mist","LifeOS"]
obs_complexity = [1,2,5,4,9]
hidden_state = [0,1,4,3,10]

plt.figure(figsize=(9,4))
plt.plot(worlds, obs_complexity, marker="o", label="observation complexity")
plt.plot(worlds, hidden_state, marker="o", label="hidden-state difficulty")
plt.ylabel("Conceptual difficulty")
plt.title("Different environments stress different RL problems")
plt.legend()
plt.xticks(rotation=15)
plt.show()

## The key transition

For Elder Mist, we control the simulator, so we know the true world state.

That lets us ask a clean experimental question:

> If I hide Sword state from the policy, what breaks?


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Controlled science</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">TRUE STATE KNOWN</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">HIDE A VARIABLE</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">TRAIN SAME POLICY</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">MEASURE DAMAGE</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
representations = ["full","no sword","no locks","minimal"]
information = [13,11,10,4]

plt.figure(figsize=(8,4))
plt.bar(representations, information)
plt.ylabel("Fields retained")
plt.title("Representation size before we even train anything")
plt.show()

## Next notebook

Now use the Elder Mist replica as a real Gymnasium-style environment.

You will manually choose actions first, then hand control to rules, Q-learning, and state-representation experiments.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Elder Mist learning sequence</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">YOU CHOOSE</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">RULE POLICY</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">Q POLICY</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">REPRESENTATION ABLATION</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>
